### Anomaly Guide Generation

In [ ]:
import torch
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from momentfm import MOMENTPipeline
import pickle
import matplotlib.cm as cm
from io import BytesIO
from PIL import Image
import json
import os

In [ ]:
os.chdir("..")
print("current path:", os.getcwd())

In [ ]:
def to_rgb(pil_image: Image.Image) -> Image.Image:
      if pil_image.mode == 'RGBA':
          white_background = Image.new("RGB", pil_image.size, (255, 255, 255))
          white_background.paste(pil_image, mask=pil_image.split()[3])  # Use alpha channel as mask
          return white_background
      else:
          return pil_image.convert("RGB")

In [ ]:
def show_ts_image(values, labels=[], xlim_arr=[0,1000], ylim_arr=[-1,1], gt=True):
    plt.close()
    plt.figure(figsize=(12, 2))
    plt.plot(
        range(xlim_arr[0], xlim_arr[1]),
        values
    )
    plt.xlim(xlim_arr)
    plt.ylim(ylim_arr)

    anomalies_idx = [xlim_arr[0]+i for i,l in enumerate(labels) if l==1] 
    if gt == True and labels != []:
        plt.bar(anomalies_idx, 2, bottom=ylim_arr[0], width=ylim_arr[1], color='green',alpha=0.5, label='Ground-truth') 

    app_range = (xlim_arr[1]-xlim_arr[0])//5
    xticks = list(range(xlim_arr[0], xlim_arr[1]+1, app_range))
    plt.xticks(xticks)
    plt.xticks(fontsize=20) 
    plt.tight_layout()
    plt.show()

In [ ]:
def segment_image_process(values, labels=[], xlim_arr=[0, 1000], ylim_arr=[-1, 1], gt=True):
    plt.close()
    plt.figure(figsize=(12, 2))
    plt.plot(
        range(xlim_arr[0], xlim_arr[1]),
        values
    )
    plt.xlim(xlim_arr)
    plt.ylim(ylim_arr)

    anomalies_idx = [xlim_arr[0]+i for i,l in enumerate(labels) if l==1] 
    if gt == True and labels != []:
        plt.bar(anomalies_idx, 2, bottom=ylim_arr[0], width=ylim_arr[1], color='green',alpha=0.5, label='Ground-truth') 

    app_range = (xlim_arr[1]-xlim_arr[0])//5
    xticks = list(range(xlim_arr[0], xlim_arr[1]+1, app_range))
    plt.xticks(xticks)
    plt.xticks(fontsize=20) 
    plt.tight_layout()

    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight') 
    buf.seek(0)
    pil_image = to_rgb(Image.open(buf))
    return pil_image

In [ ]:
def pil_show(image):
    plt.close()
    plt.imshow(image)
    plt.axis('off')
    plt.show()

In [ ]:
def pil_save(image, file_path):
    plt.close()
    plt.imshow(image)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(file_path, bbox_inches='tight', dpi=100)

In [ ]:
def find_range(binary):
    intervals = []
    in_interval = False

    for i, val in enumerate(binary):
        if val == 1 and not in_interval:
            start = i
            in_interval = True
        elif val == 0 and in_interval:
            end = i
            intervals.append((start, end))
            in_interval = False

    if in_interval:
        intervals.append((start, len(binary)))

    return intervals

In [ ]:
def get_data(data_path):
    with open(data_path, 'rb') as f:
        data = pickle.load(f)
        value_list = []
        label_list = []

        data_length = len(data['series'])

        for num in range(data_length):
            org_values = data['series'][num]
            values = [value[0] for value in org_values]

            org_answers = data['anom'][num]
            labels = [0 for _ in range(len(values))]
            for answer in org_answers[0]:
                start, end = answer[0], answer[1] # interval 
                labels[start:end] = [1 for _ in range(start, end)]

            value_list.append(values)
            label_list.append(labels)

    return value_list, label_list

In [ ]:
data_path_list = []
data_path_list.append('/home/inpyo/inpyo/Code-LMTAD-main/data/synthetic/freq/train/data.pkl')
data_path_list.append('/home/inpyo/inpyo/Code-LMTAD-main/data/synthetic/trend/train/data.pkl')
data_path_list.append('/home/inpyo/inpyo/Code-LMTAD-main/data/synthetic/point/train/data.pkl')
data_path_list.append('/home/inpyo/inpyo/Code-LMTAD-main/data/synthetic/range/train/data.pkl')

In [ ]:
key_indice = [216, 305, 379, 341]

In [ ]:
key_item_list = []
for i, data_path in enumerate(data_path_list):
    value_list, label_list = get_data(data_path)
    k = key_indice[i]
    key_item_list.append((value_list[k], label_list[k]))

In [ ]:
for key_item in key_item_list:
    value = key_item[0]
    label = key_item[1]
    show_ts_image(value, label)
    print(find_range(label))

---

In [ ]:
def time_segmentation(values, time_length, period_arr, top_k=2):
    all_segment_list = []
    for k in range(top_k):
        segment_list_size = time_length//period_arr[k]
        segment_length = period_arr[k]
        segment_list = [values[i*segment_length:i*segment_length+segment_length] for i in range(0, segment_list_size)]
        all_segment_list.append(segment_list)
    return all_segment_list

In [ ]:
class_num = 3
test_value, test_label = key_item_list[class_num][0], key_item_list[class_num][1]
test_length = len(test_value)

In [ ]:
ranges = find_range(test_label)

max_value = 0
max_idx = 0 
for i, rg in enumerate(ranges):
    curr = rg[1] - rg[0]
    if curr > max_value:
        max_idx = i
        max_value = curr
max_range = ranges[max_idx]

print(ranges)
print(max_range)

In [ ]:
ratio = 0.7
period_arr = [i for i in range(100, 1001, 100)]

rep_segment_list = []
rep_label_list = []
for period in period_arr:
    start = int(max_range[0] - (ratio*period))
    end = int(max_range[0] + ((1-ratio)*period))

    if start < 0:
        start, end = 0, period
    elif end > 1000:
        start, end = 1000-period, 1000

    rep_segment_list.append(test_value[start:end])
    rep_label_list.append(test_label[start:end])

In [ ]:
for i in range(len(period_arr)):
    rep_segment, rep_label = rep_segment_list[i], rep_label_list[i]
    show_ts_image(rep_segment, rep_label, (0, len(rep_segment)), (-1, 1))
    print(find_range(rep_label))

---

In [ ]:
def stack_images_vertically(image_list):
    widths, heights = zip(*(img.size for img in image_list))
    max_width = max(widths)
    total_height = sum(heights)
    new_image = Image.new('RGB', (max_width, total_height), (255, 255, 255))
    y_offset = 0
    for img in image_list:
        new_image.paste(img, (0, y_offset))
        y_offset += img.height
    return new_image

1. max range

In [ ]:
test_value_list = []
test_label_list = []
max_range_list = []

for class_num in range(4):
    test_value, test_label = key_item_list[class_num][0], key_item_list[class_num][1]
    test_length = len(test_value)

    # find max range
    ranges = find_range(test_label)
    max_value = 0
    max_idx = 0 
    for i, rg in enumerate(ranges):
        curr = rg[1] - rg[0]
        if curr > max_value:
            max_idx = i
            max_value = curr
    max_range = ranges[max_idx]

    test_value_list.append(test_value)
    test_label_list.append(test_label)
    max_range_list.append(max_range)
    print(class_num, max_range)

2. main segment (50~1000)

In [ ]:
period_arr = [i for i in range(50, 1001, 50)]
all_rep_segment_list = []
all_rep_label_list = []
ratio = 0.7

for i, max_range in enumerate(max_range_list):
    test_value = test_value_list[i]
    test_label = test_label_list[i]

    rep_segment_list = []
    rep_label_list = []
    for period in period_arr:
        start = int(max_range[0] - (ratio*period))
        end = int(max_range[0] + ((1-ratio)*period))

        if start < 0:
            start, end = 0, period
        elif end > 1000:
            start, end = 1000-period, 1000
        rep_segment_list.append(test_value[start:end])
        rep_label_list.append(test_label[start:end])

    all_rep_segment_list.append(rep_segment_list)
    all_rep_label_list.append(rep_label_list)

    print(i, len(rep_segment_list), len(rep_segment_list[0]), len(rep_segment_list[-1]))

3. integration

In [ ]:
json_file_path = 'Guide/anomaly_guide.json'

all_range_dict = []
for i, period in enumerate(period_arr):
    segment_list = [rep_segment_list[i] for rep_segment_list in all_rep_segment_list] 
    label_list = [rep_label_list[i] for rep_label_list in all_rep_label_list] 

    pil_image_list = []
    range_dict = {}
    range_dict['length'] = period
    range_dict['range_1'] = []
    range_dict['range_2'] = []
    range_dict['range_3'] = []
    range_dict['range_4'] = []
    range_dict['length'] = json.dumps(period)

    class_num = 0 
    for segment, label in zip(segment_list, label_list):
        pil_image = segment_image_process(segment, label, (0, period), (-1, 1))
        pil_image_list.append(pil_image)

        temp_list = []
        for rg in find_range(label):
            temp_dict = {}
            temp_dict['start'] = rg[0]
            temp_dict['end'] = rg[1]
            temp_list.append(temp_dict)
        range_dict[f'range_{class_num+1}'] = json.dumps(temp_list)
        class_num += 1

    all_range_dict.append(range_dict)
    group_image = stack_images_vertically(pil_image_list)

    file_path = f'Guide/anomaly_guide_{period}.png'
    pil_save(group_image, file_path)
    print(i, file_path)
    print(range_dict)
    pil_show(group_image)

with open(json_file_path, 'w', encoding='utf-8') as f:
      json.dump(all_range_dict, f, indent=4)

In [ ]:
guide_dict = {}
range_list = []
length = 150
with open(json_file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
    item = data[(length//50)-1]
    guide_length = json.loads(item['length'])
    for i in range(4):
        guide_range = json.loads(item[f'range_{i+1}'])
        range_list.append(guide_range)
    print(guide_length, range_list)

In [ ]:
def guide_prompt(range_list=None):
    prompt = f"""\
    You are given two images. 
    
    The first image is a visual guide showing four types of univariate time series anomalies.
    Each row corresponds to one anomaly type (from top to bottom): 
    (1) Frequency anomaly: unexpected changes in periodic patterns. Anomalous ranges: {range_list[0]} 
    (2) Trend anomaly: sudden acceleration, deceleration, or reversal. Anomalous range: {range_list[1]}
    (3) Point anomaly: individual points deviating significantly from the surrounding pattern. Anomalous range: {range_list[2]} 
    (4) Out-of-range anomaly: values that strongly deviate from the normal range. Anomalous range: {range_list[3]}. 
    Green regions in the first image indicate the anomaly ranges for each type.

    The second image shows a univariate time series split into multiple horizontal segments for compact visualization. 
    Each row corresponds to a consecutive segment of the original time series, arranged from left to right and then top to bottom.

    Based on the definitions above and your visual inspection, detect the ranges of anomalies in the second image. 
    Pay attention not only to matches with reference patterns, but also to inconsistencies or deviations that stand out compared to other segments.    
    Return the detected anomaly ranges as a list of dictionaries, one per anomaly, in terms of the x-axis coordinate.
    If there are no anomalies, return an empty list [].

    Output format only: [{{"start": ..., "end": ...}}, {{"start": ..., "end": ...}}] 
    Please do not provide any additional text or explanation.
    """
    return prompt 

In [ ]:
print(guide_prompt(range_list))

---